# Tutorial de `qbank`: ejemplos progresivos

Este cuaderno recorre los cinco ejemplos de `ejemplosManual/`, cada uno ilustrando
una mecánica nueva sobre la anterior. El flujo de trabajo es siempre el mismo:

1. Definir el problema en un fichero JSON (o en un `dict`).
2. Cargarlo con `load_problema`.
3. Iterar las variantes con `por_partes()` (vista alumno) o `por_partes_profe()` (vista revisión).
4. Exportar a AMC o Moodle cuando el problema esté listo.

**Requisito:** ejecutar las celdas en orden.

In [ ]:
from qbank import load_problema

def muestra_variantes(p, n=2, instances=1, profe=False):
    """Imprime las primeras n variantes de un problema."""
    iterador = p.por_partes_profe() if profe else p.por_partes(instances=instances)
    for i, (etiqueta, partes) in enumerate(iterador):
        if i >= n:
            break
        enunciado, cuestiones = partes[0]
        print(f"── V{etiqueta} ──")
        print(f"   {enunciado.strip()}")
        for texto, vf, activa, *resto in cuestiones:
            marca = '✓' if vf is True else ('✗' if vf is False else '·')
            print(f"   [{marca}] {texto}")
            exp = resto[0] if resto else ''
            if exp:
                print(f"       → {exp}")
        print()

---
## Ejemplo A — T/F literales, tres sublistas (`ej_a_literales.json`)

La forma más simple de `ProblemaTipo`: sin `setup`, sin variables proposicionales.
Los valores de verdad son literales `True` / `False`. Tres sublistas de 5 cuestiones
cada una generan **5 × 5 × 5 = 125 variantes** por combinación.

```json
{
  "setup": null,
  "componentes": [
    "Sea $n$ un número par y $m$ un número impar.",
    [ {"tipo": "Cuestion", "enunciado": "$n+m$ es par",   "semantica": "False", ...},
      {"tipo": "Cuestion", "enunciado": "$n+m$ es impar", "semantica": "True",  ...}, ... ],
    [ ... ],
    [ ... ]
  ]
}
```

**Mecánica:** el Marcador itera todas las combinaciones de «una cuestión por sublista».
No hay elección de enunciado: el texto del enunciado es fijo.

In [ ]:
p_a = load_problema("ejemplosManual/ej_a_literales.json")
variantes = list(p_a.por_partes())
print(f"{len(variantes)} variantes")
muestra_variantes(p_a, n=2)

---
## Ejemplo B — Supuesto en lista, dos mundos proposicionales (`ej_b_mundos.json`)

Una lista de `Supuesto` crea **mundos proposicionales** alternativos: el Marcador elige
uno y añade su semántica al mundo. Las cuestiones con `semantica: "v('crec')"` son
verdaderas en el mundo «creciente» y falsas en el mundo «decreciente», y viceversa.

```json
[
  {"tipo": "Supuesto", "enunciado": "estrictamente creciente,",
   "semantica": "v('crec')", "precond": "True"},
  {"tipo": "Supuesto", "enunciado": "estrictamente decreciente,",
   "semantica": "v('decr')", "precond": "True"}
]
```

**Mecánica:** la misma cuestión puede ser ✓ en un mundo y ✗ en otro. El número de
variantes es `(nº mundos) × (combinaciones de cuestiones)` = 2 × 3 × 3 × 4 = 72.

In [ ]:
p_b = load_problema("ejemplosManual/ej_b_mundos.json")
variantes = list(p_b.por_partes())
print(f"{len(variantes)} variantes")
# Mostramos una variante de cada mundo
muestra_variantes(p_b, n=2)

### Vista de revisión (`por_partes_profe`)

`por_partes_profe()` muestra **todas** las cuestiones de cada sublista —no solo la
elegida para el alumno— con sus marcas reales. Útil para revisar la lógica del problema.

In [ ]:
muestra_variantes(p_b, n=2, profe=True)

---
## Ejemplo C — Axioma vacío y `precond` por mundo (`ej_c_precond.json`)

Un `Supuesto` con `enunciado: ""` actúa como **axioma**: añade su fórmula al mundo
sin contribuir texto visible. Aquí establece que `v('inv') | v('sing')` siempre es
verdadero (la matriz es una cosa o la otra).

Las cuestiones usan `precond` proposicional para **aparecer solo en el mundo donde
son relevantes**: `precond: "v('inv')"` hace invisible la cuestión en el mundo `sing`
y viceversa. El Marcador descarta las combinaciones donde la precondición falla.

```json
{"tipo": "Cuestion", "enunciado": "$\\det(A) \\neq 0$",
 "semantica": "True", "precond": "v('inv')"}
```

**Diferencia con B:** en B todas las cuestiones aparecen en todos los mundos (solo
cambia su T/F). En C algunas cuestiones directamente no aparecen en ciertos mundos.

In [ ]:
p_c = load_problema("ejemplosManual/ej_c_precond.json")
variantes = list(p_c.por_partes())
print(f"{len(variantes)} variantes")
muestra_variantes(p_c, n=2, profe=True)

---
## Ejemplo D — `setup` paramétrico con interpolación `@{var}` (`ej_d_setup_vars.json`)

El campo `setup` es código Python que se ejecuta antes de cada variante. Los nombres
definidos en él quedan disponibles para interpolación con `@{nombre}` en los textos.

```python
# setup del problema
import numpy as np
a = int(np.random.randint(1, 8))
b = int(a + np.random.randint(1, 6))
diff = b - a
```

```json
{"enunciado": "$f(@{a}) < f(@{b})$", "semantica": "v('crec')"}
```

**Mecánica:** el T/F sigue siendo proposicional (depende del mundo creciente/decreciente),
pero el enunciado muestra los valores concretos de `a` y `b`. Con `instances=N` el
`setup` se ejecuta N veces con semillas distintas, multiplicando las variantes.

**`exp` también es dinámico:** admite la misma interpolación `@{var}` (o directamente
`lambda ns: ...`), así que una sola explicación sirve para todas las variantes,
mostrando los valores concretos de cada una:

```json
{"exp": "Porque @{a} < @{b} y $f$ es creciente."}
```

In [ ]:
p_d = load_problema("ejemplosManual/ej_d_setup_vars.json")
variantes = list(p_d.por_partes(instances=3))
print(f"{len(variantes)} variantes con instances=3")
muestra_variantes(p_d, n=4, instances=3)

---
## Ejemplo E — Semántica lambda e inyección numérica (`ej_e_lambda.json`)

Cuando el valor de verdad **depende de un número calculado en `setup`** (no de variables
proposicionales puras), se usa una *semántica lambda*. El `setup` precomputa fórmulas
proposicionales según el valor de `x`:

```python
# setup
x = int(np.random.randint(0, 11))
sup_u1 = v('gt_u1') if x > 3 else -v('gt_u1')
sup_u2 = v('gt_u2') if x > 7 else -v('gt_u2')
```

El `Supuesto` con lambda las inyecta en el mundo:

```json
{"tipo": "Supuesto", "enunciado": "",
 "semantica": "lambda ns: ns['sup_u1'] & ns['sup_u2']"}
```

Los axiomas iniciales establecen la jerarquía `x>u₂ ⟹ x>u₁`, que el sistema
proposicional usa para inferir respuestas coherentes aunque solo se inyecte el estado
de `x` directamente.

**Diferencia con D:** en D los números solo aparecen en el *texto*; el T/F es
proposicional. En E los números también determinan el *valor de verdad* de cada cuestión.

In [ ]:
p_e = load_problema("ejemplosManual/ej_e_lambda.json")
variantes = list(p_e.por_partes(instances=8))
print(f"{len(variantes)} variantes con instances=8")
for etiqueta, partes in variantes:
    enunciado, cuestiones = partes[0]
    x_val = enunciado.split('x = ')[1].split('.')[0].strip()
    vfs = ['✓' if vf else '✗' for _, vf, *_ in cuestiones]
    print(f"  V{etiqueta}: x={x_val}  {vfs}")

---
## Cargar un `dict` directamente (sin fichero)

`load_problema` también acepta un `dict` en memoria. Útil para probar una pregunta
en el notebook sin escribirla a disco primero.

In [ ]:
d = {
    "version": "1",
    "tipo": "ProblemaTipo",
    "nombre": "prueba_rapida",
    "seed": None,
    "setup": None,
    "componentes": [
        "Sea $p$ un número primo.",
        [
            {"tipo": "Cuestion", "enunciado": "$p$ es divisible entre $p$",
             "semantica": "True",  "precond": "True", "exp": ""},
            {"tipo": "Cuestion", "enunciado": "$p$ tiene exactamente dos divisores",
             "semantica": "True",  "precond": "True", "exp": ""},
            {"tipo": "Cuestion", "enunciado": "$p$ es par",
             "semantica": "False", "precond": "True", "exp": ""}
        ]
    ]
}

p_dict = load_problema(d)
muestra_variantes(p_dict, n=3)

---
## Exportar a AMC y Moodle

Una vez revisado el problema con `por_partes_profe()`, se exporta con las funciones
de alto nivel. Aquí exportamos el Ejemplo B a ambos formatos.

In [ ]:
import os
from qbank import AMCblock, QuizMoodle

os.makedirs("ejemplosManual/exportaciones", exist_ok=True)

# AMC (bajo nivel): generamos el bloque LaTeX para la primera variante
etiqueta, partes = list(p_b.por_partes())[0]
enunciado, cuestiones = partes[0]
bloque = AMCblock("EjemploB", etiqueta, enunciado, cuestiones)
print(bloque[:300], "...")

In [ ]:
# Moodle (alto nivel): escribe el .tex con todas las variantes
n = QuizMoodle("EjemploB", "ejemplosManual/exportaciones/", p_b, last_choice=True)
print(f"{n} variantes escritas en ejemplosManual/exportaciones/EjemploB.tex")